# System Administration Features

## Directory traversal

Write a function `count_user_dirs` that returns the number of user directories
(`/root` and `/home/*`) on a Linux system.

You can then call it before and after creating a new user to check that the
count changes.

In [ ]:
# Your code here

### Solution

In [ ]:
import pathlib


def count_user_dirs() -> int:
  home = pathlib.Path("/home")
  return sum(1 for item in home.iterdir() if item.is_dir()) + 1


print(f"Found {count_user_dirs()} user dir(s)")
!adduser --disabled-password --gecos "" new_user
print(f"Found {count_user_dirs()} user dir(s)")
!userdel -r new_user

## File Existence

Write a function `check_bashrc` that checks whether a `.bashrc` file is present in a given user's home folder. The function takes the user's name as argument and, by default (so without an argument), checks the `root` user. It returns a boolean.

In [ ]:
# Your code here

### Solution

In [ ]:
import pathlib


def check_bashrc(user: str | None = None) -> bool:
  if user is None:
    return pathlib.Path("/root/.bashrc").exists()
  return pathlib.Path(f"/home/{user}/.bashrc").exists()


check_bashrc()

## Recursive traversal of subdirectories

Count the number of files (excluding directories) contained in
the `/etc` directory and all of its subdirectories.

In [ ]:
# Your code here

### Solution

In [ ]:
import pathlib

etc_dir = pathlib.Path("/etc")

total = 0
for path in etc_dir.rglob("*"):
  if path.is_file():
    total += 1
total

In [ ]:
import pathlib

sum(1 for f in pathlib.Path("/etc").rglob("*") if f.is_file())

## Managing Permissions

Create a file `/root/.client-secret.txt` containing a random string of 20 characters (generated for instance as [advised in this SO answer](https://stackoverflow.com/a/2257449/1027951)). Create this file with only a write permission for the current user, then, once the secret is written, change these permissions to have only the read permission for the current user.

In [ ]:
# Your code here

### Solution

In [ ]:
import os
import pathlib
import random
import string

secret_value = "".join(
  random.choices(string.ascii_letters + string.digits + string.punctuation, k=20)
)
secret = pathlib.Path("/root/.client-secret.txt")
with open(secret, mode="w", opener=lambda p, f: os.open(p, f, 0o200)) as fh:
  fh.write(secret_value)
secret.chmod(0o400)

In [ ]:
!ls -al /root
!cat /root/.client-secret.txt

## Archiving Files

In this exercise, the goal is to create an archive containing all the `.conf` files of the `/etc` folder. This archive will reproduce the hierarchy of folders containing these `.conf` files (relative to `/etc`). So proceed step by step:

- List all the `.conf` files of the `/etc` folder
- Create a temporary folder
- Copy all the `.conf` files (and the folders containing them)
- Create an archive from the temporary folder

In [ ]:
# Your code here

### Solution

In [ ]:
import pathlib
import shutil
import tempfile

etc = pathlib.Path("/etc")

confs = etc.rglob("*.conf")
with tempfile.TemporaryDirectory() as tempdir:
  tempdir_path = pathlib.Path(tempdir)
  for conf in confs:
    if conf.is_file():
      temp_conf = tempdir_path / conf.relative_to(etc)
      temp_conf.parent.mkdir(parents=True, exist_ok=True)
      shutil.copy2(conf, temp_conf)
  shutil.make_archive("confs", "bztar", tempdir)

In [ ]:
# With shutil.copytree, keeps empty folders
# in the final archive
from os.path import isfile, join
from shutil import copytree


def ignore_non_confs(dirname, items):
  ignored = []
  for item in items:
    if isfile(join(dirname, item)) and not item.endswith(".conf"):
      ignored.append(item)
  return ignored


with tempfile.TemporaryDirectory(prefix="archive", delete=False) as tmp_dir:
  final_dir = pathlib.Path(tmp_dir) / "a"
  copytree("/etc", final_dir, ignore=ignore_non_confs)

## Efficient path retrieval

We will work on a corpus of Usenet newsgroup messages. Let's download it:

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!rm 20news-bydate.tar.gz

- Create a `Path` from the [`pathlib`](https://docs.python.org/3/library/pathlib.html) module representing the `20news-bydate-test` folder.
- Use the [`glob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob) method to get all the files of the corpus. How many are there?
- With the [`stat`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.stat) function, find the size of the corpus. You can compare the result with the following one:

In [ ]:
!du -sbh 20news-bydate-test/

In [ ]:
# Your code here

### Solution

In [ ]:
import pathlib

path = pathlib.Path("20news-bydate-test")
files = list(path.glob("*/*"))
print(f"Number of files: {len(files)}")

size = 0
for file in files:
  size += file.stat().st_size

# Or with sum
size = sum(file.stat().st_size for file in files)
print(f"Size of the files: {int(round(size / 1e6))}MB")

In [ ]:
!du -h 20news-bydate-test/misc.forsale/76679

## Working with file dates

Let's start by retrieving some data: log files from the `dpkg` package manager.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/dpkg-logs/logs.tar.xz
!tar xf logs.tar.xz
!rm logs.tar.xz

- Use `ls` to quickly inspect the `logs` directory. You'll see that the files have different last modification dates (as expected with log rotation).

- Create a new folder, `dated-logs`, where you'll copy the files from `logs`, renaming them following the pattern `YYYY-MM-DD_hh-mm-ss_dpkg.log.gz`, where `YYYY-MM-DD_hh-mm-ss` is the last modification date. Note in particular that you'll need to compress the uncompressed files. You can use [`gzip`](https://docs.python.org/3/library/gzip.html) for that. [`pathlib.Path.stat`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.stat) will be useful to get the files' metadata. Finally, you'll need to parse a date from a time given in seconds since the epoch (1/1/1970). You can use the [`datetime.datetime.fromtimestamp`](https://docs.python.org/3/library/datetime.html#datetime.datetime.fromtimestamp) method combined with the [`datetime.datetime.strftime`](https://docs.python.org/3/library/datetime.html#datetime.datetime.strftime) method, whose format is explained [here](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes).

In [ ]:
# Your code here

### Solution

In [ ]:
!ls -al logs

In [ ]:
import datetime
import gzip
import pathlib
import shutil


def compress(path: pathlib.Path, dest: pathlib.Path) -> None:
  with path.open(mode="rb") as fh_in, gzip.open(dest, mode="wb") as fh_out:
    shutil.copyfileobj(fh_in, fh_out)


logs_dir = pathlib.Path("logs")

dated_logs_dir = pathlib.Path("dated-logs")
dated_logs_dir.mkdir(exist_ok=True)

for item in logs_dir.iterdir():
  mtime = item.stat().st_mtime
  date = datetime.datetime.fromtimestamp(mtime, datetime.UTC)

  dest = dated_logs_dir / f"{date.strftime('%Y-%m-%d_%H-%M-%S_dpkg.log.gz')}"

  if item.suffix == ".gz":
    shutil.copy2(item, dest)
  else:
    compress(item, dest)